# Startup failures data cleaning

In [1]:
import pandas as pd
import numpy as np
import re

pd.set_option("display.max_columns", 50)

### 1.1 Load and combine the 6 sector files

In [2]:
FILES = [
    "Startup Failure (Finance and Insurance).csv",
    "Startup Failure (Food and services).csv",
    "Startup Failure (Health Care).csv",
    "Startup Failure (Manufactures).csv",
    "Startup Failure (Retail Trade).csv",
    "Startup Failures (Information Sector).csv",
]

frames = []
for f in FILES:
    d = pd.read_csv(f)
    d["source_file"] = f
    print(f"{f}: {d.shape}")
    frames.append(d)

df = pd.concat(frames, ignore_index=True)
print("\nCombined shape:", df.shape)

Startup Failure (Finance and Insurance).csv: (47, 21)
Startup Failure (Food and services).csv: (26, 18)
Startup Failure (Health Care).csv: (60, 21)
Startup Failure (Manufactures).csv: (30, 21)
Startup Failure (Retail Trade).csv: (90, 21)
Startup Failures (Information Sector).csv: (156, 21)

Combined shape: (409, 22)


### 1.2 Fix the schema mismatch
The Food and Services file has 10 flags instead of 13. It is missing `Platform Dependency`, `Toxicity/Trust Issues`, `Regulatory Pressure`, and `Overhype`, and has an extra `High Operational Costs` column. After concatenation, those cells are NaN.

- We drop `High Operational Costs`, since it exists for only 26 rows.
- We fill the missing flags with 0. **Caveat:** for the Food and Services rows, 0 means "not recorded," not "definitely absent." We don't use these 4 flags in the causal models, so this doesn't affect the results.

In [3]:
FLAG_COLS = ["Giants", "No Budget", "Competition", "Poor Market Fit",
             "Acquisition Stagnation", "Platform Dependency", "Monetization Failure",
             "Niche Limits", "Execution Flaws", "Trend Shifts",
             "Toxicity/Trust Issues", "Regulatory Pressure", "Overhype"]

print("Missing values before:\n", df.isna().sum()[df.isna().sum() > 0], "\n")

df = df.drop(columns=["High Operational Costs"])
df[FLAG_COLS] = df[FLAG_COLS].fillna(0).astype(int)

print("Missing values after:", df.isna().sum().sum())

Missing values before:
 Platform Dependency        26
Toxicity/Trust Issues      26
Regulatory Pressure        26
Overhype                   27
High Operational Costs    383
dtype: int64 

Missing values after: 0


### 1.3 Remove companies listed in more than one sector
Some companies appear in several files, sometimes with slightly different values. Keeping them would count the same company twice, so we keep the first occurrence.

In [4]:
dups = df[df["Name"].duplicated(keep=False)].sort_values("Name")
print(dups[["Name", "Sector", "Years of Operation", "How Much They Raised"]])

df = df.drop_duplicates(subset="Name", keep="first").reset_index(drop=True)
print("\nShape after removing duplicates:", df.shape)

        Name                           Sector Years of Operation  \
1    Bitpass            Finance and Insurance          2002-2008   
383  Bitpass                      Information      6 (2002-2008)   
5    Clinkle            Finance and Insurance          2011-2016   
88   Clinkle                      Health Care          2011-2016   
106  Jawbone                      Health Care          2006-2017   
149  Jawbone                    Manufacturing          2006-2017   
294  Jawbone                      Information     18 (1999-2017)   
65     Sprig  Accommodation and Food Services      4 (2013-2017)   
127    Sprig                      Health Care          2013-2017   
162     Zume                    Manufacturing          2015-2023   
378     Zume                      Information      7 (2015-2022)   

    How Much They Raised  
1                    $2M  
383                $1.5M  
5                   $30M  
88                  $30M  
106                $930M  
149                $9

### 1.4 Parse Years of Operation into founding year, end year, and lifespan (the outcome)
Two formats exist: `"3 (2010-2013)"` and `"2012-2023"`. Both contain `YYYY-YYYY`, so one regex handles both.

In [5]:
years = df["Years of Operation"].str.extract(r"(\d{4})-(\d{4})").astype(int)
df["founding_year"] = years[0]
df["end_year"] = years[1]
df["lifespan"] = df["end_year"] - df["founding_year"]

df[["Name", "Years of Operation", "founding_year", "end_year", "lifespan"]].head()

,Name,Years of Operation,founding_year,end_year,lifespan
0,Avant,2012-2023,2012,2023,11
1,Bitpass,2002-2008,2002,2008,6
2,Cake Financial,2006-2011,2006,2011,5
3,Circle,2013-2023,2013,2023,10
4,Clarity Money,2016-2022,2016,2022,6


### 1.5 Parse the funding column
Formats found in `How Much They Raised`:

| Example | Meaning | Handling |
|---|---|---|
| `$12M`, `$1.4B` | Amount raised | Convert to $ millions |
| `$5M (est.)` | Estimate | Keep the value and flag `funding_estimated = 1` |
| `$0 (Coinbase-funded)` | A product inside a big company | Flag `parent_product = 1` |
| `$1.7B (Dropbox)` | The **parent company's** total, not the product's | Flag `parent_product = 1` |
| `$10M+$403M acquisition` | Raised plus acquisition price | Keep the first number (amount raised) |
| `$lowM` | Unknown | NaN |

In [6]:
def parse_funding(s):
    """Return the first dollar amount in $ millions (B -> x1000)."""
    m = re.match(r"\$(\d+(?:\.\d+)?)\s*([MB]?)", str(s).strip())
    if not m:
        return np.nan
    value = float(m.group(1))
    return value * 1000 if m.group(2) == "B" else value

raised = df["How Much They Raised"].astype(str)
df["funding_musd"] = raised.apply(parse_funding)
df["funding_estimated"] = raised.str.contains(r"\(est\.\)").astype(int)
# A parenthesis that is NOT "(est.)" names a parent company
df["parent_product"] = raised.str.contains(r"\((?!est\.)").astype(int)

print("Could not parse:\n", df.loc[df["funding_musd"].isna(), ["Name", "How Much They Raised"]], "\n")
print("Parent-company products:", df["parent_product"].sum())
df.loc[df["parent_product"] == 1, ["Name", "How Much They Raised", "funding_musd"]].head(10)

Could not parse:
       Name How Much They Raised
91  Driver                $lowM 

Parent-company products: 46


,Name,How Much They Raised,funding_musd
6,Coinbase NFT,$0 (Coinbase-funded),0.0
29,SoFi Social,$0 (SoFi-funded),0.0
30,Square Cash,$0 (Square-funded),0.0
37,Venmo Groups,$0 (Venmo-funded),0.0
39,Wealthfront Cash,$0 (Wealthfront-funded),0.0
40,Wealthsimple Trade,$0 (Wealthsimple-funded),0.0
68,Zomato Base,$660M (Zomato),660.0
70,HelloFresh Recipe,$300M (HelloFresh),300.0
72,Tastemade Travel,$80M (Tastemade),80.0
185,Gilt Taste,$250M (Gilt),250.0


### 1.6 Build the modeling dataset
- Drop **parent-company products**. Their funding figure describes the parent, not the product, so it isn't a valid treatment value.
- Drop rows with unknown funding.
- Create the **treatment**: `high_funding = 1` if the startup raised more than the median.

In [7]:
df_model = df[(df["parent_product"] == 0) & (df["funding_musd"].notna())].copy()

median_funding = df_model["funding_musd"].median()
df_model["high_funding"] = (df_model["funding_musd"] > median_funding).astype(int)

print(f"Rows for modeling: {len(df_model)}")
print(f"Median funding: ${median_funding}M")
print("Treatment groups:", df_model["high_funding"].value_counts().to_dict())

Rows for modeling: 356
Median funding: $20.0M
Treatment groups: {0: 180, 1: 176}


### 1.7 Add more pre-treatment confounders
Sector and founding year alone leave out important differences between startups. We add business-model features from the `What They Did` column. A startup mostly chooses its business model before raising money, so these are valid confounders.

| Feature | Meaning | Why it is a confounder |
|---|---|---|
| `hardware` | Builds a physical device (wearables, drones, EVs, medical devices) | Hardware needs much more capital and has different failure dynamics |
| `physical_ops` | Runs physical operations (delivery, meal kits, subscription boxes, on-site services) | Capital intensive and thin margins that affect survival |
| `b2b` | Sells mainly to businesses or institutions | B2B startups raise differently and have stickier revenue |
| `consumer_digital` | None of the above: apps, software, e-commerce sold to consumers (the typical startup here) | Baseline group the other business models are compared to |
| `era` | Founding era: up to 2008, 2009-2014, 2015+ | Funding climates changed abruptly (dot-com, 2008 crisis, 2010s boom). Recent startups also can't have long lifespans because the data ends around 2024 |

**Method:** keyword rules on the short descriptions, then **manual overrides** after reviewing every match by hand. These labels are an approximation and a limitation of the analysis.

In [8]:
desc = df_model["What They Did"].str.lower()

HARDWARE_KW = (r"device|robot|drone|wearable|watch|tracker|camera|vehicle|\bevs?\b|cars?\b(?!-)|printer|sensor|"
               r"kiosk|lock|headband|patch|hearing aid|juicer|helmet|armband|orb\b|computers|dispenser|"
               r"stethoscope|tablets|screens|lidar|ultrasound|dialysis|blood testing|diagnostics|smartphone|biometric")
OPS_KW = (r"deliver|grocery|meal|kitchen|food chain|fast food|dining events|cleaning|furniture rental|"
          r"jewelry rental|subscription box|snack|coffee subscription|beer subscription|school meals|spas|"
          r"styling service|clothing subscription|fashion subscription|assistants")
B2B_KW = (r"for hospitals|for employees|for nursing|small biz|for small|enterprise|business|for teams|hr |hiring|"
          r"recruit|supply chain|accounts receivable|payment gateway|payment processing|underwriting|automation|"
          r"analytics|data platform|data aggregator|data processing|location data|ad platform|ad tech|marketing for|"
          r"crm|database|cloud|software|framework|encoding|backup|dashboards|for trading|restaurant ordering|tablets|"
          r"care coordination|care orchestration|for retailers|customer messaging|live chat|widget|identity|"
          r"design collaboration|prototyping|travel management|for journalists|classroom|robots$|industrial|"
          r"for industry|collaborative robots|for autonomous")

df_model["hardware"] = desc.str.contains(HARDWARE_KW).astype(int)
df_model["physical_ops"] = desc.str.contains(OPS_KW).astype(int)
df_model["b2b"] = desc.str.contains(B2B_KW).astype(int)

# Manual overrides after reviewing every match by hand
OVERRIDES = {
    "hardware": {
        "MediChain": 0,      # "lock" matched inside "blockchain"
        "BurstIQ": 0,        # same
        "Beepi": 0,          # car marketplace, not a car maker
        "Slice": 0,          # tracker *app*
        "Frontback": 0,      # camera *app*
        "Clarityn": 0,       # AI software diagnostics
        "Luminary Labs": 0,  # AI software diagnostics
    },
    "physical_ops": {
        "Augmedix": 0,       # remote scribes are B2B services, not logistics
    },
    "b2b": {
        "Anki": 0, "Cubelets": 0,   # consumer toy robots
        "IFTTT": 0,                 # mainly a consumer product
        "Augmedix": 1, "Dopamine Labs": 1, "HootSuite Media": 1,
        "Zoomdata": 1, "Spoke": 1, "Divshot": 1,
    },
}
for col, fixes in OVERRIDES.items():
    for name, value in fixes.items():
        assert name in df_model["Name"].values, name
        df_model.loc[df_model["Name"] == name, col] = value

# Startups with none of the 3 flags: consumer apps, software, and e-commerce
df_model["consumer_digital"] = (df_model[["hardware", "physical_ops", "b2b"]].sum(axis=1) == 0).astype(int)

df_model["era"] = pd.cut(df_model["founding_year"], bins=[0, 2008, 2014, 2100],
                         labels=["up_to_2008", "2009_2014", "2015_plus"]).astype(str)

for col in ["hardware", "physical_ops", "b2b", "consumer_digital"]:
    print(f"{col}: {df_model[col].sum()} startups")
print("\nEra:", df_model["era"].value_counts().to_dict())

# Review a sample of each label
for col in ["hardware", "physical_ops", "b2b", "consumer_digital"]:
    print(f"\n{col} = 1 (sample):")
    print(df_model.loc[df_model[col] == 1, ["Name", "What They Did"]].sample(5, random_state=0).to_string(index=False))

hardware: 49 startups
physical_ops: 36 startups
b2b: 60 startups
consumer_digital: 217 startups

Era: {'2009_2014': 234, 'up_to_2008': 84, '2015_plus': 38}

hardware = 1 (sample):
              Name               What They Did
       Elio Motors     Cheap three-wheeled car
          BioBeats  Stress monitoring wearable
 Butterfly Network  Handheld ultrasound device
Essential Products         Premium smartphones
 Fisker Automotive Luxury hybrid-electric cars

physical_ops = 1 (sample):
      Name                   What They Did
Trunk Club        Personal styling service
       Din Curated online grocery platform
    Tavour             Craft beer delivery
      Plum     Kids' clothing subscription
  Goldbely   Gourmet food delivery service

b2b = 1 (sample):
      Name                    What They Did
  ShopKeep          POS for small retailers
  Invision    Design collaboration platform
Drawbridge        Ad tech identity tracking
      Domo Business intelligence dashboards
  Augmedix   

### 1.8 Quick sanity checks

In [9]:
print(df_model[["lifespan", "founding_year", "funding_musd"]].describe().round(1), "\n")
print("Sector counts:\n", df_model["Sector"].value_counts(), "\n")
print("Giants flag:", df_model["Giants"].value_counts().to_dict(), "\n")

# Raw (NOT causal) difference in lifespan between the groups
print("Mean lifespan by treatment group (raw, not causal):")
print(df_model.groupby("high_funding")["lifespan"].mean().round(2))

       lifespan  founding_year  funding_musd
count     356.0          356.0         356.0
mean        7.7         2010.5         108.6
std         3.5            4.1         302.9
min         2.0         1979.0           0.4
25%         5.0         2009.0           5.0
50%         7.0         2011.0          20.0
75%        10.0         2013.0          86.5
max        18.0         2019.0        3500.0 

Sector counts:
 Sector
Information                        126
Retail Trade                        80
Health Care                         57
Finance and Insurance               41
Manufacturing                       29
Accommodation and Food Services     23
Name: count, dtype: int64 

Giants flag: {1: 274, 0: 82} 

Mean lifespan by treatment group (raw, not causal):
high_funding
0    6.51
1    8.96
Name: lifespan, dtype: float64


### 1.9 Save the cleaned data

In [ ]:
#KEEP = ["Name", "Sector", "What They Did", "founding_year", "end_year", "lifespan", "era",
        "hardware", "physical_ops", "b2b", "consumer_digital",
        "funding_musd", "funding_estimated", "high_funding"] + FLAG_COLS
#df_model[KEEP].to_csv("startups_clean.csv", index=False)
#print("Saved startups_clean.csv with shape", df_model[KEEP].shape)

Saved startups_clean.csv with shape (356, 27)
